# 2. Spatial DAGs

Spatial is a collection of hardware building blocks that are assembled into a
**directed acyclic graph** (DAG): nodes are layers or other DAGs, edges are streaming
connections. These allow you to build a versatile set of models from MLPs to Transformers.

This notebook builds a one-node DAG, emulates, estimates resources/latency,
dumps RTL, simulates, compiles with Quartus, exports a Platform Designer IP,
and saves/restores the model. Anything in this tutorial is generalizable to **ANY** DAG
and there is no limit to the number of nodes in a DAG.


In [1]:
from pathlib import Path

import numpy as np
from fpga_ai_suite.spatial.flow import run_flow
from fpga_ai_suite.spatial.layers import Directed_acyclic_graph, Dot
from fpga_ai_suite.spatial.layers.model_saver import load_model, restore_model

WORK = Path("work_dags")
WORK.mkdir(exist_ok=True)


## Building a DAG

Subclass `Directed_acyclic_graph`, store constructor parameters on `self`, then
wire ports and nodes in `init_topology()`:

- `create_input` / `create_output` — top-level streaming IO
- `add_node` — place a layer in the graph
- `connect` — wire an output (or DAG input) to an input

Do **not** name attributes `inputs` or `outputs` — those names are reserved.


In [2]:
class Dot_dag(Directed_acyclic_graph):
    """Single Dot node wrapped as a top-level DAG."""

    def __init__(self, weights, num_lanes_in=4, num_lanes_out=4):
        self.weights_data = weights
        self.num_lanes_in = num_lanes_in
        self.num_lanes_out = num_lanes_out
        super().__init__(module_name="Dot_dag")

    def init_topology(self):
        self.input = self.create_input("din", dw=16, num_lanes=self.num_lanes_in, dtype="BF16")
        self.output = self.create_output("dout", dw=16, num_lanes=self.num_lanes_out, dtype="BF16")

        self.dot = self.add_node(
            Dot(
                weights=self.weights_data,
                num_lanes_in=self.num_lanes_in,
                num_lanes_out=self.num_lanes_out,
                dtype="BF16",
                mif_init=True,
            )
        )

        self.connect(self.input, self.dot.inputs.din)
        self.connect(self.dot.outputs.dout, self.output)


# Weight matrix: features_in x features_out. Both dims must divide by lane counts.
weights = np.random.randn(64, 128).astype(np.float32)
model = Dot_dag(weights)
print(model.module_name, "nodes:", [n.module_name for n in model.dag_nodes])

Dot_dag nodes: ['dot_bf16']


## Inputs:

The inputs of a DAG feed basic layers so they must follow any requirements of the layer they're connected to. Here is a review from 1_getting_started.ipynb on shapes.

Every streaming tensor is a 3-D array:

| Axis | Meaning |
|------|---------|
| **P** | Packet count — temporal dimension; packets arrive one after another |
| **N** | Rows in the spatial vector |
| **M** | Columns in the spatial vector |

So `(1, 1, M)` is a single row vector with `M` elements.

Layer shape rules differ. Read the **`forward` docstring** on each layer:

- **Dot** expects `(P, 1, M)` (row vectors) — see `Dot.forward`
- **Transpose** / **Matmul** use full `(P, N, M)` tensors

Also: `(M) % num_lanes == 0` on every port.


In [3]:
# Dot needs (P, 1, features_in). features_in == weights.shape[-2] == 64.
P = 2
data = np.random.randn(P, 1, 64).astype(np.float32)
print("input shape (P, N, M) =", data.shape)


input shape (P, N, M) = (2, 1, 64)


## Emulation (software forward)

Unlike basic layers, DAGs require a keyword argument when inputs are passed into the forward function. 


In [4]:
# 1) Pass inputs as keyword arguments matching port names
model = Dot_dag(weights)
out_kw, = model(din=data)
print("via kwargs:", out_kw.shape)

# 2) Set IO_port.data yourself, then call forward with no arguments
model2 = Dot_dag(weights)
model2.inputs.din.data = data
out_manual, = model2()
print("via .data + forward():", out_manual.shape)
np.testing.assert_array_equal(out_kw, out_manual)

del model2

via kwargs: (2, 1, 128)
via .data + forward(): (2, 1, 128)


## `set_shape` when you do not need data

Spatial IP is shaped by tensor dimensions: hardware widths and depths come from
those shapes. Shapes must come from either `forward` (which sets them from data)
or `set_shape` (shape only). This function allows you to go through many flows (i.e., quartus_compile/save_model) without having to run an expensive forward call.

Similar to forward call, passing the shape into the set_shape function requires keyword arguments. This is the port's name followed by _shape.

Warning: You will need to create a new object if you want to change the shape after it has already been set to something different.

In [5]:
model = Dot_dag(weights)
model.set_shape(din_shape=(P, 1, 64))
print("din shape:", model.inputs.din.shape)
print("dout shape:", model.outputs.dout.shape)

# 2) Set IO_port.shape yourself, then call set_shape with no arguments
model2 = Dot_dag(weights)
model2.inputs.din.shape = (P, 1, 64)
model2.set_shape()
print("din shape (manual):", model2.inputs.din.shape)
print("dout shape (manual):", model2.outputs.dout.shape)

del model2

din shape: (2, 1, 64)
dout shape: (2, 1, 128)
din shape (manual): (2, 1, 64)
dout shape (manual): (2, 1, 128)


## Latency and resource estimates

Call these **only after** `set_shape` or `forward`.

The data_timing attribute must be set with a list of tuples (start of packet, end of packet) to estimate timing.


In [6]:
num_lanes_in = model.inputs.din.num_lanes
input_packet_size = np.prod(model.inputs.din.shape[-2:]) // num_lanes_in
# Two packets back to back 
input_timing = [(0, input_packet_size - 1), (input_packet_size, input_packet_size*2 - 1)]
model.inputs.din.data_timing = input_timing
output_timing, = model.estimate_timing()
print("Input timing (cycles): ", input_timing)
print("Output timing (cycles):", output_timing)
print("Latency in cycles (first input SOP to first output SOP)", output_timing[0][0] - input_timing[0][0])
print("resources:", model.estimate_resource_utilization())


Input timing (cycles):  [(0, 15), (16, 31)]
Output timing (cycles): [(497, 528), (1009, 1040)]
Latency in cycles (first input SOP to first output SOP) 497
resources: {'ALM': 582, 'DSP': 9, 'M20K': 10, 'M80K': 0}


## `run_flow` — dump RTL and stimulus

Call `set_shape` or `forward` before `run_flow`. For **simulation**, call
`forward` first so `dump_files/{port_name}.txt` holds the hex vectors from
emulation (that is what the RTL testbench checks against).

Modes are mutually exclusive: `sim=True`, `quartus_compile=True`, and
`save_model=True` must be separate calls.


In [7]:
# Ensure emulation data exists for dump_files/
model(din=data)

dump_dir = WORK / "output_files"

# Run flow dumps RTL and data at output_path when sim and quartus_compile are set to False
ret = run_flow(model, output_path=dump_dir)
print("Artifacts saved to", dump_dir)


Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
Running model DOT_DAG
Artifacts saved to work_dags/output_files


## `run_flow` — RTL simulation

Optional kwargs (change in this same call if you need them):

- `simulation_tool="vcs"` (default) or `"questa"`
- `dump_waveform=True` — emit waveforms (FSDB/VCD depending on tool)

Pass `gcc_path` pointing at GCC <= 13.2.0 for VCS if the default environment path is newer.


In [8]:
ret = run_flow(
    model,
    sim=True,
    recompile=True,
    # gcc_path=<path>,
    # simulation_tool="questa",  # optional alternative to default "vcs"
    # dump_waveform=True,        # optional waveforms
)


Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
Running model DOT_DAG

Running Simulation (vcs)...
Running VCS in sim_DOT_DAG (device_family=Agilex 7)
*** Using c compiler gcc instead of cc ...
                         Chronologic VCS (TM)
      Version V-2023.12-SP2-1_Full64 -- Fri Sep  4 12:30:55 2026

                    Copyright (c) 1991 - 2024 Synopsys, Inc.
   This software and the associated documentation are proprietary to Synopsys,
 Inc. This software may only be used in accordance with the terms and conditions
 of a written license agreement with Synopsys, Inc. All other use, reproduction,
   or distribution of this software is strictly prohibited.  Licensed Products
     communicate with Synopsys servers for the purpose of providing software
    updates, detecting software piracy and verifying that customers are using
    Licensed Products in conformity with the applicable License Key for such
  Licensed Products. Synopsys will us

/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: warning: rmar_nd.o: missing .note.GNU-stack section implies executable stack
/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: NOTE: This behaviour is deprecated and will be removed in a future version of the linker


rm -f _cuarc*.so _csrc*.so pre_vcsobj_*.so share_vcsobj_*.so
if [ -x ../simv ]; then chmod a-x ../simv; fi
g++  -o ../simv   -no-pie    -rdynamic  -Wl,-rpath='$ORIGIN'/simv.daidir -Wl,-rpath=./simv.daidir -Wl,-rpath=/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib -L/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib  -Wl,-rpath-link=./   objs/amcQw_d.o   _25809_archive_1.so   SIM_l.o       rmapats_mop.o rmapats.o rmar.o rmar_nd.o  rmar_llvm_0_1.o rmar_llvm_0_0.o            -lnuma -lvirsim -lerrorinf -lsnpsmalloc -lvfs    -lvcsnew -lsimprofile -lreader_common /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/libBA.a -luclinative /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_tls.o   -Wl,-whole-archive  -lvcsucli    -Wl,-no-whole-archive       ./../simv.daidir/vc_hdrs.o    /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_save_restore_new.o -ldl  -lc -lm -lpthread -ldl 
/nfs/site/disks/psg_ctools_1

## `run_flow` — Quartus compile

Depending on the device family provided, run_flow will select either A5ED065BB32AE6S for Agilex 5, or AGIB027R29A1E1VB (Agilex 7) otherwise.

In [9]:
ret = run_flow(model, quartus_compile=True, seed=1)

Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
Running model DOT_DAG

Running Quartus compile...
Info: *******************************************************************
Info: Running Quartus Prime Shell
    Info: Version 26.1.0 Build 110 03/26/2026 SC Pro Edition
    Info: Copyright (C) 2026  Altera Corporation. All rights reserved.
    Info: Your use of Altera Corporation's design tools, logic functions 
    Info: and other software and tools, and any partner logic 
    Info: functions, and any output files from any of the foregoing 
    Info: (including device programming or simulation files), and any 
    Info: associated documentation or information are expressly subject 
    Info: to the terms and conditions of the Altera Program License 
    Info: Subscription Agreement, the Altera Quartus Prime License Agreement,
    Info: the Altera IP License Agreement, or other applicable license
    Info: agreement, including, without limitation

## Platform Designer IP — `dump_ip_component`

After shapes (and usually `forward`) are set, `dump_ip_component` 
writes a relocatable PD component you can import into a
Quartus / Platform Designer project:

- `{module}_hw.tcl` — component descriptor (relative paths)
- `{module}_PD_component.sv` — PD wrapper
- `rtl/` — DAG RTL plus copied Spatial library RTL
- `dump_files/` — stimulus / weights

Requirements: top-level `axi_interface == "stream"` (the DAG default), and any
**connected** weight ports must use the DDR interface. This example keeps Dot
weights on-chip (`mif_init=True`). See `4_weight_ports.ipynb` for more details
on weight ports.


In [10]:
pd_dir = WORK / "pd_component"
model.dump_ip_component(dump_path=str(pd_dir))
print("PD component at", pd_dir)


Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
PD component at work_dags/pd_component


## Save and restore

`run_flow(..., save_model=True)` writes `model.protobuf` + `data.safetensors`.
If you saved after IO ports held data, that data is restored onto
the top-level ports. You may call `forward` / `set_shape` again on the restored model
without re-assigning inputs. You must still call `set_shape` or `forward` before
estimates or `run_flow` on the restored instance.


In [11]:
save_dir = WORK / "saved_dot_dag"
ret = run_flow(model, save_model=True, output_path=save_dir)

py_path = save_dir / "model.py"
# Creates a python file with the DAG structure of the saved model
restore_model(save_dir, py_path)
# Instantiate the restored model as an object
restored = load_model(save_dir, py_path)
print("restored class:", type(restored).__name__)

# IO data from the save is available on top-level ports when present
restored.forward()  # or restored.set_shape()
print("restored dout shape:", restored.outputs.dout.shape)
# Data consistency check
np.testing.assert_array_equal(restored.outputs.dout.data, model.outputs.dout.data)

num_lanes_in = model.inputs.din.num_lanes
input_packet_size = np.prod(model.inputs.din.shape[-2:]) // num_lanes_in
# Two packets back to back
restored.inputs.din.data_timing = [(0, input_packet_size - 1), (input_packet_size, input_packet_size*2 - 1)]

print("Output timing (cycles) after restore:", restored.estimate_timing())
print("resources after restore:", restored.estimate_resource_utilization())


Model saved to /nfs/site/disks/swuser_work_mshash/dla/external/spatial_examples/notebooks/work_dags/saved_dot_dag

Restored file saved to: /nfs/site/disks/swuser_work_mshash/dla/external/spatial_examples/notebooks/work_dags/saved_dot_dag/model.py
restored class: Dot_dag
restored dout shape: (2, 1, 128)
Output timing (cycles) after restore: ([(497, 528), (1009, 1040)],)
resources after restore: {'ALM': 582, 'DSP': 9, 'M20K': 10, 'M80K': 0}


## Extras

The cells below cover optional APIs that may help you build larger hand-built graphs:
functional `forward()` connections and `IO_bus`.



### Connections with `forward()` instead of `connect()`

If you override `forward` on the DAG class, topology is built **functionally**:

1. `init_topology()` only creates ports and `add_node(...)` — do not call `connect`.
2. After `init_topology()`, the constructor calls your `forward(*inputs)` once to
   record edges, then wires the returned ports to the DAG outputs.
3. `forward` must return a **tuple** with one entry per DAG output port.

While that recording pass runs, `add_node` returns a `Dag_layer_holder`, not the layer itself. Calling the holder (`self.dot(din)`) creates the `connect` edges, and nodes are referenced using `self.dot.node`

In [12]:
class Dot_dag_forward(Directed_acyclic_graph):
    """Same one-node Dot DAG, wired with forward() instead of connect()."""

    def __init__(self, weights, num_lanes_in=4, num_lanes_out=4):
        self.weights_data = weights
        self.num_lanes_in = num_lanes_in
        self.num_lanes_out = num_lanes_out
        super().__init__(module_name="Dot_dag_forward")

    def init_topology(self):
        self.input = self.create_input("din", dw=16, num_lanes=self.num_lanes_in, dtype="BF16")
        self.output = self.create_output("dout", dw=16, num_lanes=self.num_lanes_out, dtype="BF16")
        # Holder — not the Dot instance itself
        self.dot = self.add_node(
            Dot(
                weights=self.weights_data,
                num_lanes_in=self.num_lanes_in,
                num_lanes_out=self.num_lanes_out,
                dtype="BF16",
                mif_init=True,
            )
        )

    def forward(self, din):
        x = self.dot(din)
        return (x,)


fwd_model = Dot_dag_forward(weights)
print("add_node returned:", type(fwd_model.dot).__name__)
print("underlying layer:", type(fwd_model.dot.node).__name__)
out_fwd, = fwd_model(din=data)
print("forward-wired out shape:", out_fwd.shape)


add_node returned: Dag_layer_holder
underlying layer: Dot
forward-wired out shape: (2, 1, 128)


### Use `instance.node` for the real layer

With `forward()` wiring, attributes from `add_node` are **holders**. Anything that
needs the actual layer object must go through `.node`:

| Need | Implicit `connect` style | `forward()` style |
|------|--------------------------|-------------------|
| Call / wire in `forward` | n/a (`connect` in `init_topology`) | `self.dot(din)` (holder) |
| Layer ports | `self.dot.outputs.dout` | `self.dot.node.outputs.dout` |
| `register_rom_dependency` | `rom=self.rom` | `rom=self.rom.node` |
| Weight ports on the leaf | `self.dot.weights[...]` | `self.dot.node.weights[...]` |

### Connecting with `IO_bus`

Some layers such as Concat, Router, Merge and split use IO_bus to route IO_ports in a special way. The example below demonstrates how to connect layers with buses.

In [13]:
from fpga_ai_suite.spatial.layers import Concat, Reshape, Router


class Concat_route(Directed_acyclic_graph):
    """Routes an input then concatenates it back together"""

    def __init__(self):
        super().__init__(module_name="concat_route", use_axi_interface=False)

    def init_topology(self):
        # Using config values!
        self.din = self.create_input("din")
        self.dout = self.create_output("dout")
        self.router = self.add_node(Router(num_channels=4))
        self.concat = self.add_node(Concat(num_channels=4))
        self.reshape = self.add_node(Reshape(output_shape=(4, 1, 8)))
        # Explicit connect style:
        #   self.connect(self.din, self.router.inputs.din)
        #   self.connect(self.router.outputs.dout[0], self.concat.inputs.din[0])
        #   self.connect(self.router.outputs.dout[1], self.concat.inputs.din[1])
        #   self.connect(self.router.outputs.dout[2], self.concat.inputs.din[2])
        #   self.connect(self.router.outputs.dout[3], self.concat.inputs.din[3])
        #   self.connect(self.concat.outputs.dout, self.reshape.inputs.din)
        #   self.connect(self.reshape.outputs.dout, self.dout)

    def forward(self, din):
        y0, y1, y2, y3 = self.router(din)
        z = self.concat([y0, y1, y2, y3])
        z = self.reshape(z)
        return (z,)


bus_model = Concat_route()
in_data = np.random.randn(4, 1, 8).astype(np.float32)
out_data, = bus_model(din=in_data)
# All data gets quantized to BF16 internally.
quantized_in_data = bus_model.inputs.din.data
print("Model outs:", out_data.shape)
np.testing.assert_array_equal(out_data, quantized_in_data)


Model outs: (4, 1, 8)
